# Used Car Price Prediction — Deployment-Ready Pipeline

This version keeps preprocessing and the Random Forest model together in one scikit-learn Pipeline. The saved `car_price_pipeline.pkl` can be loaded directly by Streamlit.


In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## 1. Load and clean the dataset


In [ ]:
df = pd.read_csv("/content/Car details v3.csv")
df.head()


In [ ]:
# Remove columns that will not be used by the deployed model
df = df.drop(columns=["name", "torque"])

# Convert text measurements to numeric values.
# Missing values are intentionally kept as NaN; the Pipeline will impute them.
for col in ["engine", "max_power", "mileage"]:
    df[col] = pd.to_numeric(
        df[col].astype("string").str.extract(r"(\d+\.?\d*)", expand=False),
        errors="coerce"
    )

# The source dataset contains cars up to 2020, so age is measured relative to 2020.
REFERENCE_YEAR = 2020
df["car_age"] = REFERENCE_YEAR - df["year"]
df = df.drop(columns="year")

# Remove exact duplicate records
df = df.drop_duplicates().reset_index(drop=True)

df.info()


## 2. Define features and target

We keep categorical columns in their original form. `OneHotEncoder` will encode them inside the Pipeline. Numerical missing values are filled with the training-set median inside the Pipeline, which avoids preprocessing leakage.


In [ ]:
target = "selling_price"

cat_cols = ["fuel", "seller_type", "transmission", "owner"]
num_cols = ["km_driven", "mileage", "engine", "max_power", "seats", "car_age"]
feature_cols = num_cols + cat_cols

X = df[feature_cols].copy()
y = df[target].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)


## 3. Build preprocessing + Random Forest Pipeline


In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

pipeline


## 4. Tune the model

Because the Random Forest is inside the Pipeline, model parameters use the `model__` prefix. Cross-validation now fits preprocessing separately inside each training fold.


In [ ]:
param_grid = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 10, 20, 30, 40],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", 1.0],
}

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV R²:", random_search.best_score_)


## 5. Evaluate the final Pipeline


In [ ]:
best_pipeline = random_search.best_estimator_
y_pred = best_pipeline.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("R² =", r2)
print("MAE =", mae)
print("RMSE =", rmse)
print("MSE =", mse)


## 6. Actual vs Predicted and residual analysis


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], linestyle="--")
plt.xlabel("Actual Selling Price")
plt.ylabel("Predicted Selling Price")
plt.title("Actual vs Predicted Car Prices")
plt.show()


In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 6))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Selling Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot - Tuned Random Forest Pipeline")
plt.show()


## 7. Feature importance after preprocessing


In [ ]:
fitted_preprocessor = best_pipeline.named_steps["preprocessor"]
fitted_rf = best_pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

feature_importance = (
    pd.DataFrame({"Feature": feature_names, "Importance": fitted_rf.feature_importances_})
    .sort_values("Importance", ascending=False)
)
feature_importance.head(15)


## 8. SHAP (optional interpretation)

SHAP is applied to the transformed test data because the Random Forest inside the Pipeline receives encoded features.


In [ ]:
# If needed in Colab, run once: !pip install shap
import shap

X_shap_raw = X_test.sample(n=min(500, len(X_test)), random_state=42)
X_shap_transformed = fitted_preprocessor.transform(X_shap_raw)

# Convert sparse output if necessary for plotting.
if hasattr(X_shap_transformed, "toarray"):
    X_shap_transformed = X_shap_transformed.toarray()

X_shap_df = pd.DataFrame(X_shap_transformed, columns=feature_names, index=X_shap_raw.index)

explainer = shap.TreeExplainer(fitted_rf)
shap_values = explainer(X_shap_df)
shap.summary_plot(shap_values.values, X_shap_df)


## 9. Test one raw prediction before saving


In [ ]:
example_car = pd.DataFrame([{
    "km_driven": 60000,
    "mileage": 20.0,
    "engine": 1200.0,
    "max_power": 85.0,
    "seats": 5.0,
    "car_age": 5,
    "fuel": "Petrol",
    "seller_type": "Individual",
    "transmission": "Manual",
    "owner": "First Owner",
}])

example_prediction = best_pipeline.predict(example_car)[0]
print(f"Example predicted selling price: {example_prediction:,.0f}")


## 10. Save the complete deployment artifact

Only one model file is needed by Streamlit. It contains the imputers, one-hot encoder, and tuned Random Forest.


In [ ]:
joblib.dump(best_pipeline, "car_price_pipeline.pkl")
print("Saved: car_price_pipeline.pkl")
